Database Connection

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(find_dotenv())

def connect_to_db():
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    database = os.getenv("DB_NAME")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")

    if not all([database, user, password]):
        print("Σφάλμα: Δεν βρέθηκαν οι απαραίτητες μεταβλητές στο .env!")
        return None

    try:
        connection_uri = f"postgresql://{user}:{password}@{host}:{port}/{database}"
        engine = create_engine(connection_uri)
        with engine.connect() as connection:
            print(f"Επιτυχής σύνδεση στη βάση '{database}' στο host '{host}'!")
        return engine

    except SQLAlchemyError as e:
        print(f"Σφάλμα κατά τη σύνδεση στη ΒΔ: {e}")
        return None


print("Εκκίνηση σύνδεσης με τη βάση...")
engine = connect_to_db()

if engine is None:
    print("Τερματισμός προγράμματος λόγω αποτυχίας σύνδεσης.")
    exit(1)

Εκκίνηση σύνδεσης με τη βάση...
Επιτυχής σύνδεση στη βάση 'thesis_db' στο host 'dell-micro'!


Import Data to DF

In [5]:
import pandas as pd

query = "SELECT provider_xg, is_goal FROM statsbomb_shots_normalized"
sb_shots = pd.read_sql(query, con=engine)

Convert to boolean

In [8]:
sb_shots['is_goal'] = sb_shots['is_goal'].astype(bool)
sb_shots.dtypes
print(f"Γκολ: {len(sb_shots[sb_shots['is_goal'] == True ])} - ({sb_shots['is_goal'].mean():.2%})")
print(f"Όχι Γκολ: {(~sb_shots['is_goal']).sum()} - ({(~sb_shots['is_goal']).mean():.2%})")
print("Class Imbalanced")

Γκολ: 5465 - (9.78%)
Όχι Γκολ: 50392 - (90.22%)
Class Imbalanced


Metrics

In [9]:
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

y_true = sb_shots['is_goal']
y_pred = sb_shots['provider_xg']

print(f"Log Loss: {log_loss(y_true, y_pred):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_true, y_pred):.4f}")
print(f"Brier Score:  {brier_score_loss(y_true, y_pred):.04f}")

Log Loss: 0.2561
ROC AUC:  0.8080
Brier Score:  0.0722
